In [0]:
%sql
SHOW TABLES IN samples.wanderbricks;


#### Ejercicio 01
¿Cuántas tablas en total tiene el dataset? De las que
cargaste, ¿cuál tiene más filas? ¿Cuál tiene menos?

Tiene 16 tablas

In [0]:
from pyspark.sql.functions import avg, count, sum, round, col, desc


# Cargar las tablas
users = spark.table("samples.wanderbricks.users")
properties = spark.table("samples.wanderbricks.properties")
bookings = spark.table("samples.wanderbricks.bookings")
destinations = spark.table("samples.wanderbricks.destinations")

# Explorar users
users.printSchema()
print(f"Filas en users: {users.count()}")

# Explorar properties
properties.printSchema()
print(f"Filas en properties: {properties.count()}")

# Explorar bookings
bookings.printSchema()
print(f"Filas en bookings: {bookings.count()}")

# Explorar destinations
destinations.printSchema()
print(f"Filas en destinations: {destinations.count()}")

#### Ejercicio 02

In [0]:
properties.select("title", "property_type", "price_per_night").show(10)

In [0]:
tipos_propiedad = properties.select("property_type").distinct().count()
print(f"Tipos de propiedad diferentes: {tipos_propiedad}")

No me aparece  la columna price_per_night por lo que los demás ejercicios no los realicé ya que lo necesitan

#### Ejercicio 3

In [0]:
reservas = bookings.join(properties, "property_id")
reservas.select("booking_id", "title", "property_type", "check_in", "total_amount", "status").show(10)

In [0]:
resumen_reservas = reservas.groupBy("property_type").agg(
    count("*").alias("cantidad_reservas"),
    round(avg("total_amount"), 2).alias("monto_promedio"),
    round(sum("total_amount"), 2).alias("monto_total")
).orderBy(desc("monto_total"))

resumen_reservas.show()

#### Ejercicio 04

In [0]:
bookings_destinations = bookings.join(properties, "property_id").join(destinations, "destination_id")

resumen_destinos = bookings_destinations.groupBy(destinations["name"].alias("destino")).agg(
    count("*").alias("cantidad_reservas"),
    round(sum("total_amount"), 2).alias("monto_total")
)

# Más populares
resumen_destinos.orderBy(desc("cantidad_reservas")).show(10)

! No aparece el campo name

Ejercicio 5

In [0]:
reviews = spark.table("samples.wanderbricks.reviews").filter(col("is_deleted") == False)
reviews_properties = reviews.join(properties, "property_id")

In [0]:
mejores_propiedades = reviews_properties.groupBy("title").agg(
    count("*").alias("cantidad_resenas"),
    round(avg("rating"), 2).alias("rating_promedio")
).filter(col("cantidad_resenas") >= 3).orderBy(desc("rating_promedio"))

mejores_propiedades.show(10)

In [0]:
reviews_destinations = reviews_properties.join(destinations, "destination_id")

mejores_destinos = reviews_destinations.groupBy(destinations["name"].alias("destino")).agg(
    round(avg("rating"), 2).alias("rating_promedio")
).orderBy(desc("rating_promedio"))

mejores_destinos.show(5)

Ejercicio 06

In [0]:
%sql
SELECT 
    d.name AS destino, 
    p.property_type, 
    COUNT(b.booking_id) AS cantidad_reservas, 
    ROUND(SUM(b.total_amount), 2) AS monto_total, 
    ROUND(AVG(b.total_amount), 2) AS monto_promedio
FROM samples.wanderbricks.bookings b
JOIN samples.wanderbricks.properties p ON b.property_id = p.property_id
JOIN samples.wanderbricks.destinations d ON p.destination_id = d.destination_id
GROUP BY d.name, p.property_type
ORDER BY monto_total DESC
LIMIT 15;